# 读写文件

到目前为止，我们讨论了如何处理数据，
以及如何构建、训练和测试深度学习模型。
然而，有时我们希望保存训练的模型，
以备将来在各种环境中使用（比如在部署中进行预测）。
此外，当运行一个耗时较长的训练过程时，
最佳的做法是定期保存中间结果，
以确保在服务器电源被不小心断掉时，我们不会损失几天的计算结果。
因此，现在是时候学习如何加载和存储权重向量和整个模型了。

## (**加载和保存张量**)

对于单个张量，我们可以直接调用`load`和`save`函数分别读写它们。
这两个函数都要求我们提供一个名称，`save`要求将要保存的变量作为输入。


In [11]:
import torch
from torch import nn
from torch.nn import functional as F

x = torch.arange(4)
torch.save(x, 'x-file')

我们现在可以将存储在文件中的数据读回内存。


In [12]:
x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

我们可以[**存储一个张量列表，然后把它们读回内存。**]


In [13]:
y = torch.zeros(4)
torch.save([x, y],'x-files')
x2, y2 = torch.load('x-files')
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

我们甚至可以(**写入或读取从字符串映射到张量的字典**)。
当我们要读取或写入模型中的所有权重时，这很方便。


In [14]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

## [**加载和保存模型参数**]

保存单个权重向量（或其他张量）确实有用，
但是如果我们想保存整个模型，并在以后加载它们，
单独保存每个向量则会变得很麻烦。
毕竟，我们可能有数百个参数散布在各处。
因此，深度学习框架提供了内置函数来保存和加载整个网络。
需要注意的一个重要细节是，这将保存模型的参数而不是保存整个模型。
例如，如果我们有一个3层多层感知机，我们需要单独指定架构。
因为模型本身可以包含任意代码，所以模型本身难以序列化。
因此，为了恢复模型，我们需要用代码生成架构，
然后从磁盘加载参数。
让我们从熟悉的多层感知机开始尝试一下。


In [15]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)

接下来，我们[**将模型的参数存储在一个叫做“mlp.params”的文件中。**]


In [16]:
torch.save(net.state_dict(), 'mlp.params')

为了恢复模型，我们[**实例化了原始多层感知机模型的一个备份。**]
这里我们不需要随机初始化模型参数，而是(**直接读取文件中存储的参数。**)


In [17]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

由于两个实例具有相同的模型参数，在输入相同的`X`时，
两个实例的计算结果应该相同。
让我们来验证一下。


In [18]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

## 小结

* `save`和`load`函数可用于张量对象的文件读写。
* 我们可以通过参数字典保存和加载网络的全部参数。
* 保存架构必须在代码中完成，而不是在参数中完成。

## 练习

1. 即使不需要将经过训练的模型部署到不同的设备上，存储模型参数还有什么实际的好处？
1. 假设我们只想复用网络的一部分，以将其合并到不同的网络架构中。比如想在一个新的网络中使用之前网络的前两层，该怎么做？
1. 如何同时保存网络架构和参数？需要对架构加上什么限制？


. 存储模型参数的实际好处（即使不跨设备部署）即使始终在同一台机器或单一环境内运行，保存模型权重也是模型生命周期管理的核心步骤：容灾与断点续训（Checkpointing）： 深度学习训练可能持续数小时至数周。保存参数检查点可在遇到硬件故障、断电、内存溢出（OOM）或集群任务被抢占时，直接从最近的检查点恢复，避免从头训练。Early Stopping（早停法）与最佳状态保留： 训练过程中，测试集损失往往在某个 epoch 达到全局最优后开始过拟合。存储每个周期或历史最优的参数，可以在训练结束后回退到泛化能力最好的版本。超参数对比与实验复现： 在网格搜索或消融实验（Ablation Study）中，保留不同配置下产出的参数权重，便于横向对比表征能力、做结果复现及生成基准测试报告。下游迁移学习与特征提取： 预训练模型学到的底层通用特征（如边缘纹理、语法结构）可直接作为特征提取器或微调底座，大幅压缩后续任务的训练成本与收敛周期。2. 截取并复用网络的前两层以 PyTorch 为例，实现局部网络复用主要有两种标准方式：方法 A：直接提取子模块装配到新网络（推荐）如果原网络使用 nn.Sequential 或包含独立定义的层，可直接切片或将前两层传入新模块：Pythonimport torch
import torch.nn as nn

# 假设原网络
class OriginalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(20, 30)
        self.relu2 = nn.ReLU()
        self.layer3 = nn.Linear(30, 2)

    def forward(self, x):
        return self.layer3(self.relu2(self.layer2(self.relu1(self.layer1(x)))))

# 实例化原网络并加载权重
old_model = OriginalNet()
# old_model.load_state_dict(torch.load('old_model.pt'))

# 构建新网络，直接挂载 old_model 的前两层
class NewNet(nn.Module):
    def __init__(self, pretrained_net, freeze_backbone=True):
        super().__init__()
        # 复用前两层
        self.feature_extractor = nn.Sequential(
            pretrained_net.layer1,
            pretrained_net.relu1,
            pretrained_net.layer2,
            pretrained_net.relu2
        )
        
        # 冻结复用层的参数梯度（可选，视是否微调而定）
        if freeze_backbone:
            for param in self.feature_extractor.parameters():
                param.requires_grad = False
                
        # 新增的下游结构
        self.new_head = nn.Linear(30, 5)

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.new_head(features)

new_model = NewNet(old_model, freeze_backbone=True)
方法 B：通过 state_dict 选择性加载权重如果只想复用参数值，网络代码完全重新编写，可以使用权重键名匹配与 strict=False：Python# 提取旧模型权重中的特定层
pretrained_dict = torch.load('old_model.pt')
new_model = MyBrandNewNet()
new_model_dict = new_model.state_dict()

# 筛选出键名匹配且形状兼容的前两层权重
filtered_dict = {
    k: v for k, v in pretrained_dict.items()
    if k in new_model_dict and v.size() == new_model_dict[k].size()
}

# 更新新模型字典并加载
new_model_dict.update(filtered_dict)
new_model.load_state_dict(new_model_dict, strict=False)
3. 同时保存网络架构和参数，以及架构限制在主流框架中，保存“架构 + 参数”主要有三种技术路线：保存方式原理优势主要缺陷与架构限制Python Pickle 全对象序列化(如 torch.save(model, path))直接序列化整个 Python 对象及其类定义引用代码简单，一行调用限制极大： 强依赖工程代码路径。加载环境中必须存在同名的类定义且源码未重构，否则反序列化失败；不具备跨语言、跨版本兼容性。静态图/计算图序列化(如 TorchScript torch.jit.trace / torch.jit.script)将网络解析并固化为静态中间表示（IR）图，脱离 Python 解释器包含拓扑与权重，独立可执行，支持 C++ 运行时限制： 架构需满足静态图语法规范（见下文约束）。开放模型标准(如 ONNX)导出为标准操作算子构成的计算图真正的跨框架、跨硬件标准，生态互通限制： 只能包含 ONNX 算子集支持的操作，动态控制流受限。对网络架构的具体限制（以 TorchScript / ONNX 序列化为例）如果要实现免 Python 源码依赖的“架构 + 参数”固化保存，网络架构必须遵守以下约束：算子支持边界： 架构内使用的所有数学算子与操作，必须在框架导出器（如 TorchScript IR 或 ONNX Opset）的白名单内。包含自定义 C++/CUDA 算子或第三方未注册库函数会导致序列化报错。动态控制流限制：Tracing 模式： 仅记录单一前向传播分支。若代码中存在依赖输入张量内容的动态条件（如 if x.sum() > 0: 或根据动态阈值改变循环次数），追踪模式会永久固化第一次执行时的分支，造成逻辑丢失。Scripting 模式： 必须使用 TorchScript 静态类型子集，禁止使用任意 Python 原生语法（如复杂的反射、闭包或异构动态字典/列表）。消除隐式外部依赖： 前向传播逻辑（forward）必须是自包含的，不能依赖外部全局变量、文件系统 I/O 或未显式注册为 buffer 的状态变量。张量形状推导一致性： 尽可能避免出现不可确定的张量维度动态重塑操作（Reshape / View），以保证计算图在推断静态/动态 Batch 维度时的拓扑合法性。

[Discussions](https://discuss.d2l.ai/t/1839)
